In [1]:
import re

def char_to_digit(char):
    """
    Converts a character to its numerical equivalent according to ISO 7064.
    A=10, B=11, ..., Z=35.
    """
    if '0' <= char <= '9':
        return char
    elif 'A' <= char <= 'Z':
        # The offset is 10 for 'A' (10) and its position in the alphabet (0)
        return str(ord(char) - ord('A') + 10)
    else:
        # Handle non-alphanumeric characters if necessary (though IBANs are strict)
        raise ValueError(f"Invalid character in input: {char}")

def calculate_mod97(iban_string):
    """
    Calculates the MOD 97 checksum (result should be 1 for a valid IBAN).
    The IBAN must be provided in its "expanded" format: 
    Basic Bank Account Number (BBAN) + Country Code + Check Digits (00).
    """
    # 1. Prepare the IBAN for calculation:
    #    a. Move the first four characters (Country Code + Check Digits) to the end.
    #    b. Replace letters with digits (A=10, B=11, ... Z=35).
    #
    # Example: "GB82WEST12345698765432" -> "WEST12345698765432GB82"
    moved_iban = iban_string[4:] + iban_string[:4]
    
    # 2. Convert all characters to their digital representation
    numeric_string = "".join([char_to_digit(char) for char in moved_iban])
    
    # 3. Perform the MOD 97 calculation using the chunking method
    #    This avoids creating an enormous intermediate integer for extremely long numbers.
    
    mod_value = 0
    chunk_size = 7  # Processing in chunks of 7-9 digits is recommended
    
    # Process the numeric string in chunks
    for i in range(0, len(numeric_string), chunk_size):
        chunk = numeric_string[i : i + chunk_size]
        
        # Calculate: ((current_remainder * 10^chunk_length) + chunk_value) MOD 97
        # Since the intermediate result is only (mod_value + chunk), we avoid overflow.
        mod_value = (mod_value * (10**len(chunk)) + int(chunk)) % 97
        
    return mod_value

# ==============================================================================
# --- Example Usage ---
# ==============================================================================

# Example 1: A valid UK IBAN (The original check digits are 82)
# For the calculation, the standard is to replace the check digits with "00".
# However, to validate a full, existing IBAN, you calculate the MOD 97 result.
# A valid IBAN will yield a remainder of 1.

valid_iban = "GB82WEST12345698765432"
result_valid = calculate_mod97(valid_iban)

print(f"IBAN: {valid_iban}")
print(f"MOD 97 Remainder: {result_valid}")
print(f"Is Valid (Remainder == 1)? {'Yes' if result_valid == 1 else 'No'}\n")

# Example 2: An invalid IBAN (changed one digit)
invalid_iban = "GB82WEST12345698765433" # Changed '2' to '3' at the end
result_invalid = calculate_mod97(invalid_iban)

print(f"IBAN: {invalid_iban}")
print(f"MOD 97 Remainder: {result_invalid}")
print(f"Is Valid (Remainder == 1)? {'Yes' if result_invalid == 1 else 'No'}")

IBAN: GB82WEST12345698765432
MOD 97 Remainder: 1
Is Valid (Remainder == 1)? Yes

IBAN: GB82WEST12345698765433
MOD 97 Remainder: 28
Is Valid (Remainder == 1)? No
